# Differentiate: Modified Huron-Vidal mixrule

Example of differentiating the modified Huron-Vidal mix rule. Not heavy
explanations, just doing it.

In [ ]:
# Install if you are in Google Colab
import sys

if "google.colab" in sys.modules:
    !pip install git+https://github.com/SalvadorBrandolin/thermodiff

# You can also do: "import thermodiff as td", and access to all this things.
from thermodiff import idx_function    # Index based functions
from thermodiff import sum_components  # Sympy summation over component number
from thermodiff import sum_custom      # Sympy summation over custom limits
from thermodiff import n               # Mole number vector
from thermodiff import T               # Temperature
from thermodiff import P               # Pressure
from thermodiff import V               # Volume
from thermodiff import R               # Ideal gas constant
from thermodiff import nc              # Number of components in the mixture
from thermodiff import k, l, m         # indexes for summations
from thermodiff import DiffPlz         # Class that makes the derivatives

# We may still need sympy for some things.
import sympy as sp

# To display latext code in the notebook.
from IPython.display import display, Math

# install in colab
from thermodiff import install_in_colab

install_in_colab()

## Build the expression to differentiate

In [2]:
# Functions
ak = idx_function("a")(k, T)
b = sp.IndexedBase("b", shape=(nc,))

# i know that B and Ge are not index based, but we need lazy derivatives for
# that ones (sympy will apply chain rules otherwise)
B = idx_function("B")(n)
Dsym = idx_function("D")(n, T)
Ge = idx_function("G^E")(n, T)

nt = sum_components(n[l], l)

# Parameter
q = sp.symbols("q")

In [3]:
D = (
    R * T * B * (
        sum_components(n[k] * ak / b[k], k) + 
        1 / q * (Ge / (R * T) + sum_components(n[k] * sp.log(B / (nt * b[k])), k))
    )
)

D

R*T*(Sum(a(k, T)*n[k]/b[k], (k, 1, N_c)) + (Sum(log(B(n)/(b[k]*Sum(n[l], (l, 1, N_c))))*n[k], (k, 1, N_c)) + G^E(n, T)/(R*T))/q)*B(n)

In [4]:
diff = DiffPlz(D, [B, Dsym, ak, Ge], [k, l], "D")

## dT and dT2

In [5]:
diff.clean_plz(["dT", "dT2"])

In [6]:
diff.dt

R*T*(Sum(Derivative(a(k, T), T)*n[k]/b[k], (k, 1, N_c)) + (Derivative(G^E(n, T), T)/(R*T) - G^E(n, T)/(R*T**2))/q)*B(n) + D(n, T)/T

In [7]:
diff.dt2 = diff.dt2.simplify()

diff.dt2

(R*Sum((T*Derivative(a(k, T), (T, 2)) + 2*Derivative(a(k, T), T))*n[k]/b[k], (k, 1, N_c)) + Derivative(G^E(n, T), (T, 2))/q)*B(n)

## dTdni

because of the form of the equation, de cross derivative must have the
compositional derivative over T:

$$
    g(n, T) = RT f(n, T)
$$

Differentiating respect to $T$ and then respect to $n_i$

$$
    \frac{\partial^2 g}{\partial n_i \partial T} = R \frac{\partial f}{\partial n_i} + RT \frac{\partial^2 f}{\partial n_i \partial T}
$$

The first term is equal to:

$$
    \frac{1}{T} \frac{\partial^2 g}{\partial n_i} = R \frac{\partial f}{\partial n_i}
$$

In [8]:
diff.dtdni.has(
    diff.dni.args[0] / T + diff.dni.args[1] / T
)

True

In [9]:
# The terms with only R at the start are the ones we have to replace with
# (1 / T) * dD / dn_i
diff.dtdni.args

(R*(Sum(a(k, T)*n[k]/b[k], (k, 1, N_c)) + (Sum(log(B(n)/(b[k]*Sum(n[l], (l, 1, N_c))))*n[k], (k, 1, N_c)) + G^E(n, T)/(R*T))/q)*Derivative(B(n), n[i]),
 R*(a(i, T)/b[i] + (log(B(n)/(b[i]*Sum(n[l], (l, 1, N_c)))) + Sum((-B(n)/(b[k]*Sum(n[l], (l, 1, N_c))**2) + Derivative(B(n), n[i])/(b[k]*Sum(n[l], (l, 1, N_c))))*b[k]*n[k]*Sum(n[l], (l, 1, N_c))/B(n), (k, 1, N_c)) + Derivative(G^E(n, T), n[i])/(R*T))/q)*B(n),
 R*T*(Sum(Derivative(a(k, T), T)*n[k]/b[k], (k, 1, N_c)) + (Derivative(G^E(n, T), T)/(R*T) - G^E(n, T)/(R*T**2))/q)*Derivative(B(n), n[i]),
 R*T*(Derivative(a(i, T), T)/b[i] + (Derivative(G^E(n, T), T, n[i])/(R*T) - Derivative(G^E(n, T), n[i])/(R*T**2))/q)*B(n))

In [10]:
# I need the "i" index to manually add the derivative
from thermodiff import i

expr = diff.dtdni.args[2] + diff.dtdni.args[3] + sp.Derivative(Dsym, n[i]) / T

diff.dtdni = expr

diff.dtdni

R*T*(Derivative(a(i, T), T)/b[i] + (Derivative(G^E(n, T), T, n[i])/(R*T) - Derivative(G^E(n, T), n[i])/(R*T**2))/q)*B(n) + R*T*(Sum(Derivative(a(k, T), T)*n[k]/b[k], (k, 1, N_c)) + (Derivative(G^E(n, T), T)/(R*T) - G^E(n, T)/(R*T**2))/q)*Derivative(B(n), n[i]) + Derivative(D(n, T), n[i])/T

Check that the second term of `dtdni` is:

In [11]:
diff.dtdni.args[1]

R*T*(Sum(Derivative(a(k, T), T)*n[k]/b[k], (k, 1, N_c)) + (Derivative(G^E(n, T), T)/(R*T) - G^E(n, T)/(R*T**2))/q)*Derivative(B(n), n[i])

Is equal to the first term of `dt` over $B(n)$ multiplied by $\frac{d B}{d n_i}$
(a little reuse of terms)

In [12]:
diff.dt

R*T*(Sum(Derivative(a(k, T), T)*n[k]/b[k], (k, 1, N_c)) + (Derivative(G^E(n, T), T)/(R*T) - G^E(n, T)/(R*T**2))/q)*B(n) + D(n, T)/T

## dni

In [13]:
diff.dni

R*T*(a(i, T)/b[i] + (log(B(n)/(b[i]*Sum(n[l], (l, 1, N_c)))) + Sum((-B(n)/(b[k]*Sum(n[l], (l, 1, N_c))**2) + Derivative(B(n), n[i])/(b[k]*Sum(n[l], (l, 1, N_c))))*b[k]*n[k]*Sum(n[l], (l, 1, N_c))/B(n), (k, 1, N_c)) + Derivative(G^E(n, T), n[i])/(R*T))/q)*B(n) + R*T*(Sum(a(k, T)*n[k]/b[k], (k, 1, N_c)) + (Sum(log(B(n)/(b[k]*Sum(n[l], (l, 1, N_c))))*n[k], (k, 1, N_c)) + G^E(n, T)/(R*T))/q)*Derivative(B(n), n[i])

Checkout that the second term of `dni` is:

$$
    \frac{D}{B} \frac{d B}{d n_i}
$$

so I will replace it

In [14]:
diff.dni.args

(R*T*(Sum(a(k, T)*n[k]/b[k], (k, 1, N_c)) + (Sum(log(B(n)/(b[k]*Sum(n[l], (l, 1, N_c))))*n[k], (k, 1, N_c)) + G^E(n, T)/(R*T))/q)*Derivative(B(n), n[i]),
 R*T*(a(i, T)/b[i] + (log(B(n)/(b[i]*Sum(n[l], (l, 1, N_c)))) + Sum((-B(n)/(b[k]*Sum(n[l], (l, 1, N_c))**2) + Derivative(B(n), n[i])/(b[k]*Sum(n[l], (l, 1, N_c))))*b[k]*n[k]*Sum(n[l], (l, 1, N_c))/B(n), (k, 1, N_c)) + Derivative(G^E(n, T), n[i])/(R*T))/q)*B(n))

In [15]:
expr = diff.dni.replace(
    diff.dni.args[0],
    Dsym / B * sp.diff(B, n[i])
)

diff.dni = expr

diff.dni


R*T*(a(i, T)/b[i] + (log(B(n)/(b[i]*Sum(n[l], (l, 1, N_c)))) + Sum((-B(n)/(b[k]*Sum(n[l], (l, 1, N_c))**2) + Derivative(B(n), n[i])/(b[k]*Sum(n[l], (l, 1, N_c))))*b[k]*n[k]*Sum(n[l], (l, 1, N_c))/B(n), (k, 1, N_c)) + Derivative(G^E(n, T), n[i])/(R*T))/q)*B(n) + D(n, T)*Derivative(B(n), n[i])/B(n)

Now I will clean up all the total mole number terms:

In [16]:
diff.dni = diff.dni.replace(nt, sp.symbols(r"\textbf{n}"))

diff.dni

R*T*(a(i, T)/b[i] + (log(B(n)/(\textbf{n}*b[i])) + Sum(\textbf{n}*(Derivative(B(n), n[i])/(\textbf{n}*b[k]) - B(n)/(\textbf{n}**2*b[k]))*b[k]*n[k]/B(n), (k, 1, N_c)) + Derivative(G^E(n, T), n[i])/(R*T))/q)*B(n) + D(n, T)*Derivative(B(n), n[i])/B(n)

Is possible to distribute that $\textbf{n}$, but it's okay...

## dnidnj

The next is what happens when you not differentiate by terms:

In [17]:
diff.dnidnj

R*T*(a(i, T)/b[i] + (log(B(n)/(b[i]*Sum(n[l], (l, 1, N_c)))) + Sum((-B(n)/(b[k]*Sum(n[l], (l, 1, N_c))**2) + Derivative(B(n), n[i])/(b[k]*Sum(n[l], (l, 1, N_c))))*b[k]*n[k]*Sum(n[l], (l, 1, N_c))/B(n), (k, 1, N_c)) + Derivative(G^E(n, T), n[i])/(R*T))/q)*Derivative(B(n), n[j]) + R*T*(a(j, T)/b[j] + (log(B(n)/(b[j]*Sum(n[l], (l, 1, N_c)))) + Sum((-B(n)/(b[k]*Sum(n[l], (l, 1, N_c))**2) + Derivative(B(n), n[j])/(b[k]*Sum(n[l], (l, 1, N_c))))*b[k]*n[k]*Sum(n[l], (l, 1, N_c))/B(n), (k, 1, N_c)) + Derivative(G^E(n, T), n[j])/(R*T))/q)*Derivative(B(n), n[i]) + R*T*(Sum(a(k, T)*n[k]/b[k], (k, 1, N_c)) + (Sum(log(B(n)/(b[k]*Sum(n[l], (l, 1, N_c))))*n[k], (k, 1, N_c)) + G^E(n, T)/(R*T))/q)*Derivative(B(n), n[i], n[j]) + R*T*((-B(n)/(b[i]*Sum(n[l], (l, 1, N_c))**2) + Derivative(B(n), n[j])/(b[i]*Sum(n[l], (l, 1, N_c))))*b[i]*Sum(n[l], (l, 1, N_c))/B(n) + (-B(n)/(b[j]*Sum(n[l], (l, 1, N_c))**2) + Derivative(B(n), n[i])/(b[j]*Sum(n[l], (l, 1, N_c))))*b[j]*Sum(n[l], (l, 1, N_c))/B(n) + Sum((-B(n)/(b

Just for the laughs (not that bad):

In [18]:
diff.dnidnj.expand().simplify().replace(nt, sp.symbols(r"\textbf{n}"))

R*T*a(i, T)*Derivative(B(n), n[j])/b[i] + R*T*a(j, T)*Derivative(B(n), n[i])/b[j] + R*T*Sum((a(k, T)*Derivative(B(n), n[i], n[j])/b[k] + (log(B(n)/(\textbf{n}*b[k]))*Derivative(B(n), n[i], n[j]) + Derivative(B(n), n[i], n[j]) + Derivative(B(n), n[i])*Derivative(B(n), n[j])/B(n) - Derivative(B(n), n[i])/\textbf{n} - Derivative(B(n), n[j])/\textbf{n} + B(n)/\textbf{n}**2)/q)*n[k], (k, 1, N_c)) + R*T*log(B(n)/(\textbf{n}*b[i]))*Derivative(B(n), n[j])/q + R*T*log(B(n)/(\textbf{n}*b[j]))*Derivative(B(n), n[i])/q + R*T*Derivative(B(n), n[i])/q + R*T*Derivative(B(n), n[j])/q - 2*R*T*B(n)/(\textbf{n}*q) + B(n)*Derivative(G^E(n, T), n[i], n[j])/q + G^E(n, T)*Derivative(B(n), n[i], n[j])/q + Derivative(B(n), n[i])*Derivative(G^E(n, T), n[j])/q + Derivative(B(n), n[j])*Derivative(G^E(n, T), n[i])/q

Let's do it better from the first derivative terms:

In [19]:
Term1 = diff.dni.args[0]

Term1

D(n, T)*Derivative(B(n), n[i])/B(n)

In [20]:
Term2 = diff.dni.args[1]

Term2

R*T*(a(i, T)/b[i] + (log(B(n)/(\textbf{n}*b[i])) + Sum(\textbf{n}*(Derivative(B(n), n[i])/(\textbf{n}*b[k]) - B(n)/(\textbf{n}**2*b[k]))*b[k]*n[k]/B(n), (k, 1, N_c)) + Derivative(G^E(n, T), n[i])/(R*T))/q)*B(n)

The first term is easier:

In [21]:
from thermodiff import j

dTerm1 = sp.diff(Term1, n[j])

dTerm1

D(n, T)*Derivative(B(n), n[i], n[j])/B(n) + Derivative(B(n), n[i])*Derivative(D(n, T), n[j])/B(n) - D(n, T)*Derivative(B(n), n[i])*Derivative(B(n), n[j])/B(n)**2

The second term:

In [22]:
dTerm2 = sp.diff(Term2, n[j]).replace(nt, sp.symbols(r"\textbf{n}"))

dTerm2

R*T*(a(i, T)/b[i] + (log(B(n)/(\textbf{n}*b[i])) + Sum(\textbf{n}*(Derivative(B(n), n[i])/(\textbf{n}*b[k]) - B(n)/(\textbf{n}**2*b[k]))*b[k]*n[k]/B(n), (k, 1, N_c)) + Derivative(G^E(n, T), n[i])/(R*T))/q)*Derivative(B(n), n[j]) + R*T*(Sum(\textbf{n}*(Derivative(B(n), n[i])/(\textbf{n}*b[k]) - B(n)/(\textbf{n}**2*b[k]))*KroneckerDelta(j, k)*b[k]/B(n) - \textbf{n}*(Derivative(B(n), n[i])/(\textbf{n}*b[k]) - B(n)/(\textbf{n}**2*b[k]))*Derivative(B(n), n[j])*b[k]*n[k]/B(n)**2 + \textbf{n}*(Derivative(B(n), n[i], n[j])/(\textbf{n}*b[k]) - Derivative(B(n), n[j])/(\textbf{n}**2*b[k]))*b[k]*n[k]/B(n), (k, 1, N_c)) + Derivative(B(n), n[j])/B(n) + Derivative(G^E(n, T), n[i], n[j])/(R*T))*B(n)/q

Check that the first term of the `dTerm2` is equal to the first term of `dni`
over $B$ multiplied by $\frac{dB}{d n_j}$

In [23]:
diff.dni

R*T*(a(i, T)/b[i] + (log(B(n)/(\textbf{n}*b[i])) + Sum(\textbf{n}*(Derivative(B(n), n[i])/(\textbf{n}*b[k]) - B(n)/(\textbf{n}**2*b[k]))*b[k]*n[k]/B(n), (k, 1, N_c)) + Derivative(G^E(n, T), n[i])/(R*T))/q)*B(n) + D(n, T)*Derivative(B(n), n[i])/B(n)

So we can express that first term of `dTerm2` in terms of the first derivative
of $D$ as:

$$
    \frac{\frac{\partial D}{\partial n_i} - \frac{D \frac{d B}{d n_i}}{B}}{B} \frac{d B}{d n_j}
$$

In [24]:
dTerm2 = dTerm2.replace(
    dTerm2.args[0],
    (sp.diff(Dsym, n[i]) - Dsym * sp.diff(B, n[i]) / B) / B * sp.diff(B, n[j])
)

dTerm2

R*T*(Sum(\textbf{n}*(Derivative(B(n), n[i])/(\textbf{n}*b[k]) - B(n)/(\textbf{n}**2*b[k]))*KroneckerDelta(j, k)*b[k]/B(n) - \textbf{n}*(Derivative(B(n), n[i])/(\textbf{n}*b[k]) - B(n)/(\textbf{n}**2*b[k]))*Derivative(B(n), n[j])*b[k]*n[k]/B(n)**2 + \textbf{n}*(Derivative(B(n), n[i], n[j])/(\textbf{n}*b[k]) - Derivative(B(n), n[j])/(\textbf{n}**2*b[k]))*b[k]*n[k]/B(n), (k, 1, N_c)) + Derivative(B(n), n[j])/B(n) + Derivative(G^E(n, T), n[i], n[j])/(R*T))*B(n)/q + (Derivative(D(n, T), n[i]) - D(n, T)*Derivative(B(n), n[i])/B(n))*Derivative(B(n), n[j])/B(n)

then we have the ugly Kroneckers. Lets see if I can take them out by hand:

In [25]:
import thermodiff as td

dTerm2 = td.handle_sum_kronecker(dTerm2, j)

dTerm2

R*T*(\textbf{n}*(Derivative(B(n), n[i])/(\textbf{n}*b[j]) - B(n)/(\textbf{n}**2*b[j]))*b[j]/B(n) + Sum(\textbf{n}*(Derivative(B(n), n[i], n[j])/(\textbf{n}*b[k]) - Derivative(B(n), n[j])/(\textbf{n}**2*b[k]))*b[k]*n[k]/B(n), (k, 1, N_c)) + Sum(-\textbf{n}*(Derivative(B(n), n[i])/(\textbf{n}*b[k]) - B(n)/(\textbf{n}**2*b[k]))*Derivative(B(n), n[j])*b[k]*n[k]/B(n)**2, (k, 1, N_c)) + Derivative(B(n), n[j])/B(n) + Derivative(G^E(n, T), n[i], n[j])/(R*T))*B(n)/q + (Derivative(D(n, T), n[i]) - D(n, T)*Derivative(B(n), n[i])/B(n))*Derivative(B(n), n[j])/B(n)

In [26]:
diff.dnidnj = dTerm1 + dTerm2

diff.dnidnj = diff.dnidnj.simplify()

diff.dnidnj

R*T*(Derivative(B(n), n[i], n[j]) - Derivative(B(n), n[i])*Derivative(B(n), n[j])/B(n))*Sum(n[k], (k, 1, N_c))/q + R*T*Derivative(B(n), n[i])/q + R*T*Derivative(B(n), n[j])/q - R*T*B(n)/(\textbf{n}*q) + D(n, T)*Derivative(B(n), n[i], n[j])/B(n) + Derivative(B(n), n[i])*Derivative(D(n, T), n[j])/B(n) + Derivative(B(n), n[j])*Derivative(D(n, T), n[i])/B(n) - 2*D(n, T)*Derivative(B(n), n[i])*Derivative(B(n), n[j])/B(n)**2 + B(n)*Derivative(G^E(n, T), n[i], n[j])/q

In [27]:
diff.dnidnj = diff.dnidnj.replace(sum_components(n[k], k), sp.symbols(r"\textbf{n}"))

diff.dnidnj

R*T*\textbf{n}*(Derivative(B(n), n[i], n[j]) - Derivative(B(n), n[i])*Derivative(B(n), n[j])/B(n))/q + R*T*Derivative(B(n), n[i])/q + R*T*Derivative(B(n), n[j])/q - R*T*B(n)/(\textbf{n}*q) + D(n, T)*Derivative(B(n), n[i], n[j])/B(n) + Derivative(B(n), n[i])*Derivative(D(n, T), n[j])/B(n) + Derivative(B(n), n[j])*Derivative(D(n, T), n[i])/B(n) - 2*D(n, T)*Derivative(B(n), n[i])*Derivative(B(n), n[j])/B(n)**2 + B(n)*Derivative(G^E(n, T), n[i], n[j])/q

## Clean Latex

In [28]:
latex = diff.latex_readable_plz()

derivs = ["dT", "dT2", "dTn", "dni", "dn2"]

for d in derivs:
    print(f"{d}:")
    display(Math(latex[d]))

dT:


<IPython.core.display.Math object>

dT2:


<IPython.core.display.Math object>

dTn:


<IPython.core.display.Math object>

dni:


<IPython.core.display.Math object>

dn2:


<IPython.core.display.Math object>

In [29]:
latex = diff.latex_readable_plz()

derivs = ["dT", "dT2", "dTn", "dni", "dn2"]

for d in derivs:
    print(f"{d}:")
    print(latex[d])

dT:
B R T \left(\sum_{k=1}^{N_{c}} \frac{\frac{\partial a_{k}}{\partial T} {n}_{k}}{{b}_{k}} + \frac{- \frac{G^{E}}{R T^{2}} + \frac{\frac{\partial G^{E}}{\partial T}}{R T}}{q}\right) + \frac{D{\left(n,T \right)}}{T}
dT2:
B \left(R \sum_{k=1}^{N_{c}} \frac{\left(T \frac{\partial^2 a_{k}}{\partial T^2} + 2 \frac{\partial a_{k}}{\partial T}\right) {n}_{k}}{{b}_{k}} + \frac{\frac{\partial^2 G^{E}}{\partial T^2}}{q}\right)
dTn:
B R T \left(\frac{\frac{\partial a_{i}}{\partial T}}{{b}_{i}} + \frac{\frac{\frac{\partial^2 G^{E}}{\partial T \partial n_i}}{R T} - \frac{\frac{\partial G^{E}}{\partial n_i}}{R T^{2}}}{q}\right) + R T \frac{\partial B}{\partial n_i} \left(\sum_{k=1}^{N_{c}} \frac{\frac{\partial a_{k}}{\partial T} {n}_{k}}{{b}_{k}} + \frac{- \frac{G^{E}}{R T^{2}} + \frac{\frac{\partial G^{E}}{\partial T}}{R T}}{q}\right) + \frac{\frac{\partial D}{\partial n_i}}{T}
dni:
B R T \left(\frac{a_{i}}{{b}_{i}} + \frac{\log{\left(\frac{B}{\textbf{n} {b}_{i}} \right)} + \sum_{k=1}^{N_{c}} \fr